# GRPO Demo (Local Version)

This tutorial demonstrates training the [Gemma](https://deepmind.google/models/gemma/) 
2 2B-IT model on the [GSM8K math reasoning benchmark](https://huggingface.co/datasets/openai/gsm8k) 
using [Group Relative Policy Optimization (GRPO)](https://arxiv.org/pdf/2402.03300). 

**This is a localized version** that can run on GCP TPU without Kaggle-specific dependencies.

## Requirements
- TPU v5e-8 (recommended)
- Kaggle account with Gemma model access
- (Optional) Weights & Biases account


## Configuration

**IMPORTANT: Fill in your credentials before running!**


In [1]:
# ============================================================
# USER CONFIGURATION - FILL IN BEFORE RUNNING
# ============================================================

# Kaggle credentials (required for downloading Gemma model)
# Get from: https://www.kaggle.com/settings -> API -> Create New Token
KAGGLE_USERNAME = "yhechen"  # <- Change this
KAGGLE_KEY = "KGAT_30945005a87a250883619d211d5c16ad"        # <- Change this

# Wandb configuration (optional)
# Set to True to enable experiment tracking
USE_WANDB = True
WANDB_API_KEY = "205549d0058f815ada2b4dfe68285341cbbc36b3"      # <- Change if USE_WANDB=True

# ============================================================
# Apply configuration
# ============================================================
import os

# Set Kaggle credentials
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

# Set Wandb mode
if USE_WANDB:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
else:
    os.environ['WANDB_DISABLED'] = 'true'
    os.environ['WANDB_MODE'] = 'offline'

print("Configuration applied!")
print(f"  Kaggle Username: {KAGGLE_USERNAME}")
print(f"  Wandb Enabled: {USE_WANDB}")


Configuration applied!
  Kaggle Username: yhechen
  Wandb Enabled: True


## Imports


In [ ]:
import functools
import gc
import os
from pprint import pprint
import re
import csv
import shutil

from flax import nnx
import grain
import humanize
import jax
import jax.numpy as jnp
import kagglehub
import optax
from orbax import checkpoint as ocp
from pathlib import Path
import qwix
import tensorflow_datasets as tfds
from tqdm.auto import tqdm
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.rollout import base_rollout
from tunix.sft import metrics_logger

if USE_WANDB:
    import wandb


## Hyperparameters


In [ ]:
# ====== Data ======
TRAIN_DATA_DIR = "./data/train"
TEST_DATA_DIR = "./data/test"
TRAIN_FRACTION = 1.0

# ====== LoRA ======
RANK = 64
ALPHA = 64.0

# ====== Sharding ======
MESH = [(1, 4), ("fsdp", "tp")]

# ====== GRPO ======
MAX_PROMPT_LENGTH = 256
TOTAL_GENERATION_STEPS = 512
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = 50
NUM_GENERATIONS = 4

# === other GRPO configs ===
NUM_ITERATIONS = 1
BETA = 0.08
EPSILON = 0.2

# ====== Training ======
TRAIN_MICRO_BATCH_SIZE = 2
NUM_BATCHES = 3738
NUM_TEST_BATCHES = 100
EVAL_EVERY_N_STEPS = 10
NUM_EPOCHS = 1

MAX_STEPS = int(NUM_BATCHES * NUM_ITERATIONS * TRAIN_FRACTION * NUM_EPOCHS)

# === AdamW, warmup, cosine scheduler ===
LEARNING_RATE = 3e-6
B1 = 0.9
B2 = 0.99
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 0.1 * MAX_STEPS
MAX_GRAD_NORM = 0.1

# Checkpoint saving
INTERMEDIATE_CKPT_DIR = "/tmp/content/intermediate_ckpt/"
CKPT_DIR = "/tmp/content/ckpts/"
SAVE_INTERVAL_STEPS = 500
MAX_TO_KEEP = 4

# ====== Inference ======
GENERATION_CONFIGS = {
    "greedy": {"temperature": 1e-4, "top_k": 1, "top_p": 1.0},
    "standard": {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
    "liberal": {"temperature": 0.85, "top_k": 2000, "top_p": 1.0},
}

print(f"Total training steps: {MAX_STEPS}")
print(f"Warmup steps: {WARMUP_STEPS}")


In [ ]:
def show_hbm_usage():
    """Displays memory usage per device."""
    fmt_size = functools.partial(humanize.naturalsize, binary=True)
    for d in jax.local_devices():
        stats = d.memory_stats()
        used = stats["bytes_in_use"]
        limit = stats["bytes_limit"]
        print(f"Using {fmt_size(used)} / {fmt_size(limit)} ({used/limit:%}) on {d}")


## Data preprocessing


In [ ]:
reasoning_start = "<reasoning>"
reasoning_end = "</reasoning>"
solution_start = "<answer>"
solution_end = "</answer>"

SYSTEM_PROMPT = f"""You are given a problem. Think about the problem and \
provide your reasoning. Place it between {reasoning_start} and \
{reasoning_end}. Then, provide the final answer (i.e., just one numerical \
value) between {solution_start} and {solution_end}."""

TEMPLATE = """<start_of_turn>user
{system_prompt}

{question}<end_of_turn>
<start_of_turn>model"""


In [ ]:
def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

def download_kaggle_dataset(target_dir="./data/gsm8k"):
    os.makedirs(target_dir, exist_ok=True)
    src = kagglehub.dataset_download("thedevastator/grade-school-math-8k-q-a")
    src = Path(src)
    dst = Path(target_dir)
    for csv_file in src.glob("*.csv"):
        shutil.copy2(csv_file, dst / csv_file.name)
        print(f"Copied {csv_file.name} -> {dst/csv_file.name}")
    return target_dir

def get_dataset(data_dir, split="train", source="tfds") -> grain.MapDataset:
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    if source == "tfds":
        import tensorflow_datasets.text.gsm8k
        data = tfds.data_source(
            "gsm8k",
            split=split,
            data_dir=data_dir,
            builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
            download=True,
        )
    elif source == "kaggle":
        kaggle_dir = download_kaggle_dataset(data_dir)
        file_name = "main_" + split + ".csv"
        csv_path = os.path.join(kaggle_dir, file_name)
        data = []
        with open(csv_path, newline="", encoding="utf-8") as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                data.append({"question": row["question"], "answer": row["answer"]})
    else:
        raise ValueError(f"Unknown source: {source}")

    def _as_text(v):
        return v if isinstance(v, str) else v.decode("utf-8")

    dataset = (
        grain.MapDataset.source(data)
        .shuffle(seed=42)
        .map(lambda x: {
            "prompts": TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=_as_text(x["question"])),
            "question": _as_text(x["question"]),
            "answer": extract_hash_answer(_as_text(x["answer"])),
        })
    )
    return dataset


In [ ]:
# Load dataset
source = 'kaggle'
print(f"Using data source: {source}")

dataset = get_dataset(TRAIN_DATA_DIR, "train", source).batch(TRAIN_MICRO_BATCH_SIZE)[:NUM_BATCHES]

if TRAIN_FRACTION == 1.0:
    train_dataset = dataset.repeat(NUM_EPOCHS)
    val_dataset = None
else:
    train_dataset = dataset[: int(len(dataset) * TRAIN_FRACTION)]
    train_dataset = train_dataset.repeat(NUM_EPOCHS)
    val_dataset = dataset[int(len(dataset) * TRAIN_FRACTION) :].repeat(NUM_EPOCHS)

test_dataset = get_dataset(TEST_DATA_DIR, "test", source).batch(TRAIN_MICRO_BATCH_SIZE)[:NUM_TEST_BATCHES]

dataset_lengths = (len(train_dataset), len(val_dataset) if val_dataset else 0, len(test_dataset))
print(f"Dataset contains {dataset_lengths} batches (train, val, test)")


## Load model


In [ ]:
# Download model from Kaggle
model_path = {"gemma2": "google/gemma-2/flax/"}
model_family = "gemma2"
model_version = "gemma2-2b-it"
print(f"Downloading model: {model_path[model_family]}{model_version}")

kaggle_ckpt_path = kagglehub.model_download(f"{model_path[model_family]}{model_version}")
print(f"Model downloaded to: {kaggle_ckpt_path}")


In [ ]:
# Convert checkpoint to NNX format
!rm -rf /tmp/content/intermediate_ckpt/*
!rm -rf /tmp/content/ckpts/*

if model_family == "gemma2":
    params = params_lib.load_and_format_params(os.path.join(kaggle_ckpt_path, "gemma2-2b-it"))
    gemma = gemma_lib.Transformer.from_params(params, version="2-2b-it")
    checkpointer = ocp.StandardCheckpointer()
    _, state = nnx.split(gemma)
    checkpointer.save(os.path.join(INTERMEDIATE_CKPT_DIR, "state"), state)
    checkpointer.wait_until_finished()
    del params, gemma, state
    gc.collect()
    print("Model checkpoint saved!")


In [ ]:
def get_gemma_ref_model(ckpt_path):
    mesh = jax.make_mesh(*MESH)
    model_config = gemma_lib.ModelConfig.gemma2_2b()
    abs_gemma = nnx.eval_shape(lambda: gemma_lib.Transformer(model_config, rngs=nnx.Rngs(params=0)))
    abs_state = nnx.state(abs_gemma)
    abs_state = jax.tree.map(
        lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
        abs_state, nnx.get_named_sharding(abs_state, mesh)
    )
    checkpointer = ocp.StandardCheckpointer()
    restored_params = checkpointer.restore(ckpt_path, target=abs_state)
    graph_def, _ = nnx.split(abs_gemma)
    gemma = nnx.merge(graph_def, restored_params)
    return gemma, mesh, model_config

def get_lora_model(base_model, mesh):
    lora_provider = qwix.LoraProvider(
        module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj|.*attn_vec_einsum",
        rank=RANK, alpha=ALPHA
    )
    model_input = base_model.get_model_input()
    lora_model = qwix.apply_lora_to_model(base_model, lora_provider, **model_input)
    with mesh:
        state = nnx.state(lora_model)
        pspecs = nnx.get_partition_spec(state)
        sharded_state = jax.lax.with_sharding_constraint(state, pspecs)
        nnx.update(lora_model, sharded_state)
    return lora_model


In [ ]:
# Load reference model
ref_model, mesh, model_config = get_gemma_ref_model(os.path.join(INTERMEDIATE_CKPT_DIR, "state"))
print("Reference model loaded!")

# Load policy model with LoRA
lora_policy = get_lora_model(ref_model, mesh=mesh)
print("Policy model (with LoRA) loaded!")

# Load tokenizer
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=os.path.join(kaggle_ckpt_path, "tokenizer.model"))
print("Tokenizer loaded!")


## Reward functions


In [ ]:
match_format = re.compile(
    rf"^[\s]{{0,}}{reasoning_start}.+?{reasoning_end}.*?{solution_start}(.+?){solution_end}[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)

match_numbers = re.compile(rf"{solution_start}.*?([\d\.]{{1,}})", flags=re.MULTILINE | re.DOTALL)

def match_format_exactly(prompts, completions, **kwargs):
    return [0 if match_format.search(r) is None else 3.0 for r in completions]

def match_format_approximately(prompts, completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        score += 0.5 if completion.count(reasoning_start) == 1 else -0.5
        score += 0.5 if completion.count(reasoning_end) == 1 else -0.5
        score += 0.5 if completion.count(solution_start) == 1 else -0.5
        score += 0.5 if completion.count(solution_end) == 1 else -0.5
        scores.append(score)
    return scores

def check_answer(prompts, completions, answer, **kwargs):
    extracted = [guess.group(1) if (guess := match_format.search(r)) else None for r in completions]
    scores = []
    for guess, true_answer in zip(extracted, answer):
        if guess is None:
            scores.append(0)
            continue
        score = 0
        if guess == true_answer:
            score += 3.0
        elif guess.strip() == true_answer.strip():
            score += 1.5
        else:
            try:
                ratio = float(guess) / float(true_answer)
                if 0.9 <= ratio <= 1.1:
                    score += 0.5
                elif 0.8 <= ratio <= 1.2:
                    score += 0.25
                else:
                    score -= 1.0
            except:
                score -= 0.5
        scores.append(score)
    return scores

def check_numbers(prompts, completions, answer, **kwargs):
    question = kwargs["question"]
    extracted = [guess.group(1) if (guess := match_numbers.search(r)) else None for r in completions]
    scores = []
    print(f"Q: {question[0][:50]}... | A: {answer[0]} | Extracted: {extracted[0]}")
    for guess, true_answer in zip(extracted, answer):
        if guess is None:
            scores.append(0)
            continue
        try:
            scores.append(1.5 if float(guess.strip()) == float(true_answer.strip()) else 0.0)
        except:
            scores.append(0)
    return scores


In [ ]:
def generate(question, sampler, temperature=0.7, top_k=50, top_p=0.95, seed=None):
    if isinstance(question, str):
        input_batch = [TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=question)]
    else:
        input_batch = [TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=q) for q in question]
    out_data = sampler(input_strings=input_batch, max_generation_steps=768,
                       temperature=temperature, top_k=top_k, top_p=top_p, echo=False, seed=seed)
    return out_data.text[0] if isinstance(question, str) else out_data.text

def evaluate(dataset, sampler, temperature=0.7, top_k=50, top_p=0.95, num_passes=1):
    corr, partially_corr, corr_format, total = 0, 0, 0, 0
    for batch in tqdm(dataset):
        answers, questions = batch["answer"], batch["question"]
        for question, answer in zip(questions, answers):
            responses = [generate(question, sampler, temperature, top_k, top_p, seed=p) for p in range(num_passes)]
            found_corr, found_partial, found_format = False, False, False
            for response in responses:
                extracted = match_numbers.search(response)
                extracted = extracted.group(1) if extracted else "-1000000"
                try:
                    if float(extracted.strip()) == float(answer.strip()):
                        found_corr = True
                    ratio = float(extracted.strip()) / float(answer.strip())
                    if 0.9 <= ratio <= 1.1:
                        found_partial = True
                except:
                    pass
                if match_format.search(response):
                    found_format = True
            corr += int(found_corr)
            partially_corr += int(found_partial)
            corr_format += int(found_format)
            total += 1
            if total % 20 == 0:
                print(f"Progress: {total}, Acc: {corr/total*100:.1f}%, Format: {corr_format/total*100:.1f}%")
    return corr, total, corr/total*100, partially_corr/total*100, corr_format/total*100


## Pre-training evaluation


In [ ]:
sampler = sampler_lib.Sampler(
    transformer=lora_policy,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

print("Evaluating pre-training performance...")
corr, total, accuracy, partial_accuracy, format_accuracy = evaluate(
    test_dataset, sampler, **GENERATION_CONFIGS["greedy"]
)
print(f"\n=== Pre-training Results ===")
print(f"Accuracy: {accuracy:.2f}% | Partial: {partial_accuracy:.2f}% | Format: {format_accuracy:.2f}%")


## Training setup


In [ ]:
# Checkpoint and logging options
checkpointing_options = ocp.CheckpointManagerOptions(save_interval_steps=SAVE_INTERVAL_STEPS, max_to_keep=MAX_TO_KEEP)
metrics_logging_options = metrics_logger.MetricsLoggerOptions(log_dir="/tmp/content/tmp/tensorboard/grpo", flush_every_n_steps=20)

# Optimizer
optimizer = optax.adamw(
    learning_rate=optax.schedules.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=LEARNING_RATE, warmup_steps=WARMUP_STEPS, decay_steps=MAX_STEPS, end_value=0.0
    ),
    b1=B1, b2=B2, weight_decay=WEIGHT_DECAY
)
if MAX_GRAD_NORM:
    optimizer = optax.chain(optax.clip_by_global_norm(max_norm=MAX_GRAD_NORM), optimizer)

# Training config
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine='vanilla',
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=EVAL_EVERY_N_STEPS,
        max_steps=MAX_STEPS,
        mini_batch_size=TRAIN_MICRO_BATCH_SIZE,
        train_micro_batch_size=TRAIN_MICRO_BATCH_SIZE,
        metrics_logging_options=metrics_logging_options,
        checkpoint_root_directory=CKPT_DIR,
        checkpointing_options=checkpointing_options,
    ),
    rollout_config=base_rollout.RolloutConfig(
        max_tokens_to_generate=TOTAL_GENERATION_STEPS,
        max_prompt_length=MAX_PROMPT_LENGTH,
        kv_cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
    ),
)

grpo_config = GRPOConfig(num_generations=NUM_GENERATIONS, num_iterations=NUM_ITERATIONS, beta=BETA, epsilon=EPSILON)


In [ ]:
# Create trainer
rl_cluster = rl_cluster_lib.RLCluster(
    actor=lora_policy,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

grpo_trainer = GRPOLearner(
    rl_cluster=rl_cluster,
    reward_fns=[match_format_exactly, match_format_approximately, check_answer, check_numbers],
    grpo_config=grpo_config,
)
print("GRPO Trainer initialized!")
